<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/04_construction_review_indicators_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การตรวจรูปแบบโครงการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

ผลลัพธ์ใช้สำหรับจัดลำดับการตรวจเอกสาร ไม่ใช่หลักฐานว่ามีการทุจริตหรือแบ่งซื้อแบ่งจ้าง

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(style='whitegrid', font='TH Sarabun New')

plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
MUTED = '#667085'
GRID = '#E4E7EC'

In [ ]:
processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

figure_dir = processed_dir.parents[3] / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

project_path = processed_dir / 'construction_projects_2569.csv'
contract_path = processed_dir / 'construction_contracts_2569.csv'

project_data = pd.read_csv(project_path, low_memory=False)
contract_data = pd.read_csv(contract_path, low_memory=False)

In [ ]:
project_id_column = 'รหัสโครงการ'
project_name_column = 'ชื่อโครงการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
subagency_column = 'ชื่อหน่วยงานย่อย'
province_column = 'จังหวัด'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
budget_column = 'วงเงินงบประมาณ (บาท)'
awarded_price_column = 'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
transaction_date_column = 'วันที่เกิดรายการ'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'

specific_method_value = 'เฉพาะเจาะจง'
legal_budget_ceiling = 500_000

project_data['in_study_scope'] = (
    project_data[method_column].eq(specific_method_value)
    & project_data[budget_column].le(legal_budget_ceiling)
)

study_project_data = project_data.loc[
    project_data['in_study_scope']
].copy()

study_project_ids = set(
    study_project_data[project_id_column]
)

study_summary = pd.Series({
    'โครงการก่อสร้างทั้งหมด': len(project_data),
    'โครงการในกลุ่มศึกษา': len(study_project_data),
    'สัดส่วนโครงการทั้งหมด (%)': (
        len(study_project_data) / len(project_data) * 100
    )
}, name='value')

display(study_summary.to_frame())

In [ ]:
project_supplier_data = (
    contract_data
    .groupby(
        [project_id_column, supplier_id_column]
    )
    .agg(
        supplier_name=(supplier_name_column, 'first'),
        supplier_awarded_value=(contract_value_column, 'sum')
    )
    .reset_index()
    .merge(
        project_data[
            [
                project_id_column,
                project_name_column,
                agency_column,
                subagency_column,
                province_column,
                method_column,
                budget_column,
                awarded_price_column,
                transaction_date_column,
                'in_study_scope'
            ]
        ],
        on=project_id_column,
        how='left'
    )
)

## 1. Pattern 1 — โครงการใกล้เพดานที่เกิดซ้ำ

ตรวจโครงการวิธีเฉพาะเจาะจง วงเงิน 490,000–500,000 บาท ที่มีหน่วยงานย่อย ผู้รับจ้าง และวันที่เกิดรายการเดียวกันอย่างน้อย 3 โครงการ

In [ ]:
valid_cluster_rows = (
    project_supplier_data[subagency_column].notna()
    & project_supplier_data[supplier_id_column].notna()
    & project_supplier_data[transaction_date_column].notna()
    & project_supplier_data[transaction_date_column].ne('-')
)

near_ceiling_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & valid_cluster_rows
    & project_supplier_data[budget_column].between(
        490_000,
        legal_budget_ceiling,
        inclusive='both'
    )
].copy()

cluster_columns = [
    subagency_column,
    supplier_id_column,
    transaction_date_column
]

near_ceiling_clusters = (
    near_ceiling_data
    .groupby(cluster_columns, dropna=False)
    .agg(
        supplier_name=('supplier_name', 'first'),
        project_count=(project_id_column, 'nunique'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

repeated_clusters = (
    near_ceiling_clusters.loc[
        near_ceiling_clusters['project_count'].ge(3)
    ]
    .sort_values(
        ['project_count', 'total_budget'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern1_project_ids = set(
    near_ceiling_data
    .merge(
        repeated_clusters[cluster_columns],
        on=cluster_columns,
        how='inner',
        validate='many_to_many'
    )[project_id_column]
)

project_data['flag_pattern_1'] = (
    project_data[project_id_column].isin(pattern1_project_ids)
)

pattern1_summary = pd.Series({
    'กลุ่มที่ผ่านเกณฑ์': len(repeated_clusters),
    'โครงการที่เกี่ยวข้อง': project_data['flag_pattern_1'].sum(),
    'สัดส่วนกลุ่มศึกษา (%)': (
        project_data['flag_pattern_1'].sum()
        / len(study_project_data)
        * 100
    )
}, name='value')

display(pattern1_summary.to_frame())
display(repeated_clusters.head(15))

### ผล Pattern 1

พบ 1,244 กลุ่ม รวม 5,613 โครงการ

## 2. Pattern 2 — หน่วยงานพึ่งพาผู้รับจ้างรายเดียวสูง

ตรวจผู้รับจ้างที่ได้รับอย่างน้อย 10 โครงการ และครองทั้งจำนวนโครงการและมูลค่าอย่างน้อย 75% ภายในหน่วยงานย่อย

In [ ]:
study_supplier_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & project_supplier_data[subagency_column].notna()
    & project_supplier_data[supplier_id_column].notna()
].copy()

supplier_relationships = (
    study_supplier_data
    .groupby(
        [subagency_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=('supplier_name', 'first'),
        supplier_project_count=(project_id_column, 'nunique'),
        supplier_awarded_value=('supplier_awarded_value', 'sum')
    )
    .reset_index()
)

subagency_totals = (
    study_project_data
    .groupby(subagency_column, dropna=False)
    .agg(
        subagency_project_count=(project_id_column, 'nunique'),
        subagency_awarded_value=(awarded_price_column, 'sum')
    )
    .reset_index()
)

supplier_relationships = supplier_relationships.merge(
    subagency_totals,
    on=subagency_column,
    how='left',
    validate='many_to_one'
)

supplier_relationships['project_share_pct'] = (
    supplier_relationships['supplier_project_count']
    / supplier_relationships['subagency_project_count']
    * 100
)

supplier_relationships['value_share_pct'] = (
    supplier_relationships['supplier_awarded_value']
    / supplier_relationships['subagency_awarded_value']
    * 100
)

display(
    supplier_relationships
    .sort_values('supplier_project_count', ascending=False)
    .head(15)
)

In [ ]:
high_dependence_relationships = (
    supplier_relationships.loc[
        supplier_relationships['supplier_project_count'].ge(10)
        & supplier_relationships['project_share_pct'].ge(75)
        & supplier_relationships['value_share_pct'].ge(75)
    ]
    .sort_values(
        ['project_share_pct', 'supplier_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern2_project_ids = set(
    study_supplier_data
    .merge(
        high_dependence_relationships[
            [subagency_column, supplier_id_column]
        ],
        on=[subagency_column, supplier_id_column],
        how='inner',
        validate='many_to_many'
    )[project_id_column]
)

project_data['flag_pattern_2'] = (
    project_data[project_id_column].isin(pattern2_project_ids)
)

pattern2_summary = pd.Series({
    'คู่หน่วยงาน–ผู้รับจ้างที่ผ่านเกณฑ์': len(high_dependence_relationships),
    'ผู้รับจ้างที่เกี่ยวข้อง': (
        high_dependence_relationships[supplier_id_column].nunique()
    ),
    'โครงการที่เกี่ยวข้อง': project_data['flag_pattern_2'].sum()
}, name='value')

display(pattern2_summary.to_frame())
display(high_dependence_relationships.head(15))

### ผล Pattern 2

พบ 142 คู่หน่วยงานย่อย–ผู้รับจ้าง รวม 2,421 โครงการ

## 3. Pattern 3 — การพึ่งพาผู้รับจ้างภายในจังหวัด

ตรวจว่ามีผู้รับจ้างรายใดครองทั้งจำนวนโครงการและมูลค่าตั้งแต่ 50% ภายในจังหวัดหรือไม่ โดยพิจารณาจังหวัดที่มีอย่างน้อย 20 โครงการและผู้รับจ้างที่ได้รับอย่างน้อย 5 โครงการ

In [ ]:
valid_geographic_data = study_supplier_data.loc[
    study_supplier_data[province_column].notna()
    & ~study_supplier_data[province_column]
    .astype('string')
    .str.strip()
    .isin(['', '-', 'ไม่ระบุ'])
].copy()

province_supplier_relationships = (
    valid_geographic_data
    .groupby(
        [province_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=('supplier_name', 'first'),
        supplier_project_count=(project_id_column, 'nunique'),
        supplier_awarded_value=('supplier_awarded_value', 'sum')
    )
    .reset_index()
)

province_totals = (
    study_project_data.loc[
        study_project_data[province_column].notna()
        & ~study_project_data[province_column]
        .astype('string')
        .str.strip()
        .isin(['', '-', 'ไม่ระบุ'])
    ]
    .groupby(province_column, dropna=False)
    .agg(
        province_project_count=(project_id_column, 'nunique'),
        province_awarded_value=(awarded_price_column, 'sum')
    )
    .reset_index()
)

province_supplier_relationships = (
    province_supplier_relationships
    .merge(
        province_totals,
        on=province_column,
        how='left',
        validate='many_to_one'
    )
)

province_supplier_relationships['project_share_pct'] = (
    province_supplier_relationships['supplier_project_count']
    / province_supplier_relationships['province_project_count']
    * 100
)

province_supplier_relationships['value_share_pct'] = (
    province_supplier_relationships['supplier_awarded_value']
    / province_supplier_relationships['province_awarded_value']
    * 100
)

display(
    province_supplier_relationships
    .sort_values('project_share_pct', ascending=False)
    .head(15)
)

In [ ]:
eligible_province_relationships = (
    province_supplier_relationships.loc[
        province_supplier_relationships['province_project_count'].ge(20)
        & province_supplier_relationships['supplier_project_count'].ge(5)
    ]
    .copy()
)

geographic_dominance_relationships = (
    eligible_province_relationships.loc[
        eligible_province_relationships['project_share_pct'].ge(50)
        & eligible_province_relationships['value_share_pct'].ge(50)
    ]
    .copy()
)

geographic_summary = pd.Series({
    'คู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์': (
        len(geographic_dominance_relationships)
    ),
    'ส่วนแบ่งจำนวนโครงการสูงสุด (%)': (
        eligible_province_relationships['project_share_pct'].max()
    ),
    'ส่วนแบ่งมูลค่าสูงสุด (%)': (
        eligible_province_relationships['value_share_pct'].max()
    )
}, name='value')

display(geographic_summary.to_frame())

### ผล Pattern 3

ไม่พบคู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์ ส่วนแบ่งจำนวนโครงการสูงสุดอยู่ที่ 14.45% และส่วนแบ่งมูลค่าสูงสุดอยู่ที่ 15.12%

## 4. โครงการที่เข้าเงื่อนไขทั้งสอง Pattern

In [ ]:
pattern1_context = (
    near_ceiling_data
    .merge(
        repeated_clusters[
            cluster_columns + ['project_count']
        ].rename(columns={
            'project_count': 'cluster_project_count'
        }),
        on=cluster_columns,
        how='inner',
        validate='many_to_one'
    )
)

pattern2_context = (
    study_supplier_data
    .merge(
        high_dependence_relationships[
            [
                subagency_column,
                supplier_id_column,
                'supplier_project_count',
                'subagency_project_count',
                'project_share_pct',
                'value_share_pct'
            ]
        ],
        on=[subagency_column, supplier_id_column],
        how='inner',
        validate='many_to_one'
    )
)

priority_context = (
    pattern1_context
    .merge(
        pattern2_context[
            [
                project_id_column,
                subagency_column,
                supplier_id_column,
                'supplier_project_count',
                'subagency_project_count',
                'project_share_pct',
                'value_share_pct'
            ]
        ],
        on=[
            project_id_column,
            subagency_column,
            supplier_id_column
        ],
        how='inner',
        validate='many_to_one'
    )
)

priority_project_ids = set(
    priority_context[project_id_column]
)

study_review_data = project_data.loc[
    project_data['in_study_scope']
].copy()

study_review_data['flag_pattern_1'] = (
    study_review_data[project_id_column]
    .isin(pattern1_project_ids)
)

study_review_data['flag_pattern_2'] = (
    study_review_data[project_id_column]
    .isin(pattern2_project_ids)
)

study_review_data['priority_review'] = (
    study_review_data[project_id_column]
    .isin(priority_project_ids)
)

overlap_summary = pd.Series({
    'เกิดซ้ำใกล้เพดาน': study_review_data['flag_pattern_1'].sum(),
    'หน่วยงานพึ่งพาผู้รับจ้างรายเดียวสูง': (
        study_review_data['flag_pattern_2'].sum()
    ),
    'เข้าเงื่อนไขทั้งสอง Pattern': (
        study_review_data['priority_review'].sum()
    ),
    'เข้าอย่างน้อยหนึ่ง Pattern': (
        study_review_data['flag_pattern_1']
        | study_review_data['flag_pattern_2']
    ).sum(),
    'ไม่เข้า Pattern': (
        ~study_review_data['flag_pattern_1']
        & ~study_review_data['flag_pattern_2']
    ).sum()
}, name='project_count')

display(overlap_summary.to_frame())

In [ ]:
plot_data = pd.Series({
    'เกิดซ้ำใกล้เพดาน\nหน่วยงาน–ผู้รับจ้าง–วันเดียวกัน': (
        study_review_data['flag_pattern_1'].sum()
    ),
    'หน่วยงานพึ่งพา\nผู้รับจ้างรายเดียวสูง': (
        study_review_data['flag_pattern_2'].sum()
    ),
    'เข้าเงื่อนไข\nทั้งสอง Pattern': (
        study_review_data['priority_review'].sum()
    )
})

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    plot_data.index,
    plot_data.values,
    color=[BLUE, BLUE, ORANGE],
    height=0.58
)

ax.bar_label(
    bars,
    labels=[f'{value:,.0f}' for value in plot_data.values],
    padding=5,
    fontsize=11,
    color=TEXT
)

ax.invert_yaxis()
ax.set_xlim(0, plot_data.max() * 1.18)
ax.set_title(
    'จำนวนโครงการที่เข้าเงื่อนไขการตรวจสอบ',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.tight_layout()

png_path = figure_dir / 'fig04_01_review_conditions.png'
svg_path = figure_dir / 'fig04_01_review_conditions.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

In [ ]:
priority_support = (
    priority_context[
        [
            project_id_column,
            subagency_column,
            supplier_id_column,
            'supplier_name',
            transaction_date_column,
            'cluster_project_count',
            'supplier_project_count',
            'subagency_project_count',
            'project_share_pct',
            'value_share_pct'
        ]
    ]
)

priority_projects = (
    project_data.loc[
        project_data[project_id_column].isin(priority_project_ids),
        [
            project_id_column,
            project_name_column,
            agency_column,
            subagency_column,
            province_column,
            budget_column,
            awarded_price_column,
            transaction_date_column
        ]
    ]
    .merge(
        priority_support,
        on=[
            project_id_column,
            subagency_column,
            transaction_date_column
        ],
        how='left',
        validate='one_to_one'
    )
    .sort_values(
        [
            'cluster_project_count',
            'project_share_pct',
            budget_column
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

priority_groups = (
    priority_context
    .groupby(
        [
            subagency_column,
            supplier_id_column,
            'supplier_name',
            transaction_date_column
        ],
        dropna=False
    )
    .agg(
        priority_project_count=(project_id_column, 'nunique'),
        cluster_project_count=('cluster_project_count', 'max'),
        supplier_project_count=('supplier_project_count', 'max'),
        subagency_project_count=('subagency_project_count', 'max'),
        project_share_pct=('project_share_pct', 'max'),
        value_share_pct=('value_share_pct', 'max')
    )
    .reset_index()
    .sort_values(
        [
            'priority_project_count',
            'cluster_project_count',
            'project_share_pct'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(priority_groups.head(15))
display(
    priority_projects[
        [
            project_id_column,
            project_name_column,
            subagency_column,
            'supplier_name',
            transaction_date_column,
            budget_column,
            'cluster_project_count',
            'project_share_pct',
            'value_share_pct'
        ]
    ].head(20)
)

print(f'กลุ่มตรวจสอบ: {len(priority_groups):,}')
print(f'โครงการตรวจสอบลำดับแรก: {len(priority_projects):,}')

### การใช้ผลลัพธ์

291 โครงการไม่ได้หมายถึง 291 โครงการทุจริต แต่เป็นโครงการที่มีเหตุผลให้เปิดเอกสารก่อน เพราะพบพร้อมกันสองเงื่อนไข:

1. อยู่ในกลุ่มโครงการใกล้ 500,000 บาทของหน่วยงาน ผู้รับจ้าง และวันเดียวกัน
2. หน่วยงานนั้นพึ่งพาผู้รับจ้างรายดังกล่าวสูงทั้งจำนวนโครงการและมูลค่า

ควรตรวจเป็นกลุ่มตามหน่วยงาน–ผู้รับจ้าง–วันที่เกิดรายการ โดยดูขอบเขตงาน สถานที่ ระยะเวลาอนุมัติ ผู้เสนอราคา และเอกสารเสนอราคา เพื่อแยกว่าเป็นหลายโครงการที่มีเหตุผลรองรับหรือเป็นงานที่ควรพิจารณาร่วมกัน

## 5. สรุป

- โครงการเกิดซ้ำใกล้เพดาน 5,613 โครงการ
- โครงการที่หน่วยงานพึ่งพาผู้รับจ้างรายเดียวสูง 2,421 โครงการ
- โครงการที่เข้าเงื่อนไขทั้งสอง Pattern 291 โครงการ
- ไม่พบการครองงานระดับจังหวัดตามเกณฑ์ 50%
- การตรวจต่อควรเริ่มจากกลุ่มที่มีจำนวนโครงการเกิดซ้ำมาก แล้วเปิดเอกสารของโครงการในกลุ่มพร้อมกัน

## 6. บันทึกผลลัพธ์

In [ ]:
project_flags = project_data[
    [project_id_column, 'in_study_scope']
].copy()

project_flags['flag_pattern_1'] = (
    project_flags[project_id_column].isin(pattern1_project_ids)
)

project_flags['flag_pattern_2'] = (
    project_flags[project_id_column].isin(pattern2_project_ids)
)

project_flags['priority_review'] = (
    project_flags[project_id_column].isin(priority_project_ids)
)

output_objects = {
    'project_review_indicators_2569.csv': project_flags,
    'priority_review_groups_2569.csv': priority_groups,
    'priority_review_projects_2569.csv': priority_projects,
    'repeated_near_500k_clusters_2569.csv': repeated_clusters,
    'high_supplier_dependence_2569.csv': high_dependence_relationships,
    'province_supplier_summary_2569.csv': province_supplier_relationships
}

export_records = []

for file_name, output_data in output_objects.items():
    output_path = processed_dir / file_name
    output_data.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig'
    )
    export_records.append({
        'file_name': file_name,
        'rows': len(output_data),
        'file_size_mb': output_path.stat().st_size / 1024**2
    })

export_summary = pd.DataFrame(export_records)
display(export_summary)